In [1]:
import numpy as np
import cv2
import open3d as o3d

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
import cv2
import numpy as np

disp = cv2.imread("/home/aadi_iiith/Desktop/IS/OpenStereo/data/KITTI15/training/disp_occ_0/000000_10.png", cv2.IMREAD_GRAYSCALE)
disp = cv2.resize(disp, (1242, 375))

disp_fake = cv2.imread("depth.png", cv2.IMREAD_GRAYSCALE)
disp_fake = cv2.resize(disp_fake, (1242, 375))

# Concatenate side by side
combined = np.hstack((disp, disp_fake))

print("disp shape:", disp.shape, "dtype:", disp.dtype)
print("disp_fake shape:", disp_fake.shape, "dtype:", disp_fake.dtype)

cv2.imshow("Disparity (left) | Fake Disparity (right)", combined)
cv2.waitKey(0)
cv2.destroyAllWindows()


disp shape: (375, 1242) dtype: uint8
disp_fake shape: (375, 1242) dtype: uint8


In [3]:
depth = disp*(0.53/7.215377e+02)

In [69]:
img = cv2.imread("/home/aadi_iiith/Desktop/IS/OpenStereo/data/KITTI15/training/image_2/000001_10.png",cv2.IMREAD_UNCHANGED)
print(img.shape)

(375, 1242, 3)


In [70]:

# K_fake = np.array([
#     9.597910e+02, 0.000000e+00, 6.960217e+02,
#     0.000000e+00, 9.569251e+02, 2.241806e+02,
#     0.000000e+00, 0.000000e+00, 1.000000e+00
# ], dtype=np.float32).reshape(3, 3)


# rgbd_ref = o3d.geometry.RGBDImage.create_from_color_and_depth(
#     o3d.geometry.Image(img),
#     o3d.geometry.Image(depth.astype(np.float32)),
#     depth_scale=1e-3, depth_trunc=3000.0, convert_rgb_to_intensity=False)
# intrinsic = o3d.camera.PinholeCameraIntrinsic()
# intrinsic.intrinsic_matrix = K_fake
# pcd_fake = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd_ref, intrinsic)

In [71]:

R = np.array([
    [9.998817e-01,  1.511453e-02, -2.841595e-03],
    [-1.511724e-02, 9.998853e-01, -9.338510e-04],
    [2.827154e-03,  9.766976e-04,  9.999955e-01]
])

P = np.array([
    [7.215377e+02, 0.000000e+00, 6.095593e+02, 4.485728e+01],
    [0.000000e+00, 7.215377e+02, 1.728540e+02, 2.163791e-01],
    [0.000000e+00, 0.000000e+00, 1.000000e+00, 2.745884e-03]
])


In [72]:
import numpy as np

def recover_K_from_P_and_R(P, R):
    """
    Given P (3x4) and R (3x3), return intrinsic K (3x3).
    Handles the common cases where R is a rotation (uses R.T),
    otherwise uses np.linalg.inv(R).
    Ensures positive diagonal in K and normalizes K[2,2]=1.
    """
    P = np.asarray(P, dtype=float)
    R = np.asarray(R, dtype=float)
    if P.shape != (3, 4):
        raise ValueError("P must be shape (3,4)")
    if R.shape != (3, 3):
        raise ValueError("R must be shape (3,3)")

    M = P[:, :3]  # left 3x3 block

    # Use R.T if R is (almost) orthonormal; otherwise use inverse
    if np.allclose(R @ R.T, np.eye(3), atol=1e-6):
        Rinv = R.T
    else:
        Rinv = np.linalg.inv(R)

    K_raw = M @ Rinv

    # Fix sign on diagonal: enforce positive diagonal entries
    diag_sign = np.sign(np.diag(K_raw))
    # replace zeros with +1 (in case any diag element is exactly 0)
    diag_sign[diag_sign == 0] = 1.0
    S = np.diag(diag_sign)
    K_signed = K_raw @ S
    # Optionally adjust R as well: R_corrected = S @ R   (not returned here)

    # Normalize so that K[2,2] == 1
    K = K_signed / K_signed[2, 2]

    return K

# Example usage:
# P = np.array([...]).reshape(3,4)
# R = np.array([...]).reshape(3,3)
K = recover_K_from_P_and_R(P, R)
print(K)


[[ 7.19723460e+02 -1.14769478e+01  6.11599207e+02]
 [ 1.04145690e+01  7.21296766e+02  1.73558727e+02]
 [-2.84160779e-03 -9.33855202e-04  1.00000000e+00]]


In [85]:

disp = cv2.imread("/home/aadi_iiith/Desktop/IS/OpenStereo/data/KITTI15/training/disp_occ_0/000001_10.png",cv2.IMREAD_GRAYSCALE)
disp = cv2.resize(disp,(1242,375))
depth  = (4.485728e+01)/disp   
rgbd_ref = o3d.geometry.RGBDImage.create_from_color_and_depth(
    o3d.geometry.Image(img),
    o3d.geometry.Image(depth.astype(np.float32)),
    depth_scale=1e-3, depth_trunc=3000.0, convert_rgb_to_intensity=False)
intrinsic = o3d.camera.PinholeCameraIntrinsic()
intrinsic.intrinsic_matrix = K
pcd_ref = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd_ref, intrinsic)

disp_fake = cv2.imread("depth.png",cv2.IMREAD_GRAYSCALE)

disp_fake = cv2.resize(disp_fake,(1242,375))
print(disp_fake.min(),disp_fake.max())
print(disp.min(),disp.max())

disp_fake = disp_fake.astype(np.float32)
disp_fake = disp_fake*0.75
disp_fake[:disp_fake.shape[0]//3,:] = 0
depth_fake = (4.485728e+01)/disp_fake   
     

rgbd_fake = o3d.geometry.RGBDImage.create_from_color_and_depth(
    o3d.geometry.Image(img),
    o3d.geometry.Image(depth_fake.astype(np.float32)*1e-3),
    depth_scale=1 , depth_trunc=300.0, convert_rgb_to_intensity=False)
intrinsic = o3d.camera.PinholeCameraIntrinsic()
intrinsic.intrinsic_matrix = K
pcd_fake = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd_fake, intrinsic)
print(disp.dtype,disp_fake.dtype)

3 238
0 78
uint8 float32


/tmp/ipykernel_324152/3905607586.py:3: RuntimeWarning: divide by zero encountered in divide
  depth  = (4.485728e+01)/disp
/tmp/ipykernel_324152/3905607586.py:21: RuntimeWarning: divide by zero encountered in divide
  depth_fake = (4.485728e+01)/disp_fake


In [86]:
o3d.visualization.draw_geometries([pcd_fake])

In [82]:
o3d.visualization.draw_geometries([pcd_ref])

In [ ]:
norm = cv2.normalize(disp, None, 0, 255, cv2.NORM_MINMAX)

# Apply colormap (COLORMAP_JET is a common heatmap style)
heatmap = cv2.applyColorMap(norm.astype(np.uint8), cv2.COLORMAP_JET)

cv2.imwrite("heatmap.png", heatmap)

True

In [13]:
import numpy as np


def disp_to_color(disp, max_disp=None):
    """
    Transfer disparity map to color map
    Args:
        disp (numpy.array): disparity map in (Height, Width) layout, value range [0, 255]
        max_disp (int): max disparity, optionally specifies the scaling factor
    Returns:
        disparity color map (numpy.array): disparity map in (Height, Width, 3) layout, range [0,255]
    """
    h, w = disp.shape

    if max_disp is None:
        max_disp = np.max(disp)

    # scale the disp to [0,1] by max_disp
    disp = disp / max_disp

    # reshape the disparity to [Height*Width, 1]
    disp = disp.reshape((h * w, 1))

    # convert to color map, with shape [Height*Width, 3]
    disp = disp_map(disp)

    # convert to RGB-mode
    disp = disp.reshape((h, w, 3))
    disp = disp * 255.0

    disp = disp.clip(0, 255)

    return disp


def disp_map(disp):
    """
    Based on color histogram, convert the gray disp into color disp map.
    The histogram consists of 7 bins, value of each is e.g. [114.0, 185.0, 114.0, 174.0, 114.0, 185.0, 114.0]
    Accumulate each bin, named cbins, and scale it to [0,1], e.g. [0.114, 0.299, 0.413, 0.587, 0.701, 0.886, 1.0]
    For each value in disp, we have to find which bin it belongs to
    Therefore, we have to compare it with every value in cbins
    Finally, we have to get the ratio of it accounts for the bin, and then we can interpolate it with the histogram map
    For example, 0.780 belongs to the 5th bin, the ratio is (0.780-0.701)/0.114,
    then we can interpolate it into 3 channel with the 5th [0, 1, 0] and 6th [0, 1, 1] channel-map
    Inputs:
        disp: numpy array, disparity gray map in (Height * Width, 1) layout, value range [0,1]
    Outputs:
        disp: numpy array, disparity color map in (Height * Width, 3) layout, value range [0,1]
    """
    bin_map = np.array([
        [0, 0, 0, 114],
        [0, 0, 1, 185],
        [1, 0, 0, 114],
        [1, 0, 1, 174],
        [0, 1, 0, 114],
        [0, 1, 1, 185],
        [1, 1, 0, 114],
        [1, 1, 1, 0]
    ])
    # grab the last element of each column and convert into float type, e.g. 114 -> 114.0
    # the final result: [114.0, 185.0, 114.0, 174.0, 114.0, 185.0, 114.0]
    bins = bin_map[0:bin_map.shape[0] - 1, bin_map.shape[1] - 1].astype(float)

    # reshape the bins from [7] into [7,1]
    bins = bins.reshape((bins.shape[0], 1))

    # accumulate element in bins, and get [114.0, 299.0, 413.0, 587.0, 701.0, 886.0, 1000.0]
    cbins = np.cumsum(bins)

    # divide the last element in cbins, e.g. 1000.0
    bins = bins / cbins[cbins.shape[0] - 1]

    # divide the last element of cbins, e.g. 1000.0, and reshape it, final shape [6,1]
    cbins = cbins[0:cbins.shape[0] - 1] / cbins[cbins.shape[0] - 1]
    cbins = cbins.reshape((cbins.shape[0], 1))

    # transpose disp array, and repeat disp 6 times in axis-0, 1 times in axis-1, final shape=[6, Height*Width]
    ind = np.tile(disp.T, (6, 1))
    tmp = np.tile(cbins, (1, disp.size))

    # get the number of disp's elements bigger than  each value in cbins, and sum up the 6 numbers
    b = (ind > tmp).astype(int)
    s = np.sum(b, axis=0)

    bins = 1 / bins

    # add an element 0 ahead of cbins, [0, cbins]
    t = cbins
    cbins = np.zeros((cbins.size + 1, 1))
    cbins[1:] = t

    # get the ratio and interpolate it
    disp = (disp - cbins[s]) * bins[s]
    disp = bin_map[s, 0:3] * np.tile(1 - disp, (1, 3)) + bin_map[s + 1, 0:3] * np.tile(disp, (1, 3))

    return disp

img = disp_to_color(disp=disp,max_disp=192)
cv2.imwrite("heatmap.png",img)

[ WARN:0@248.133] global loadsave.cpp:1063 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.


True

In [63]:
from PIL import Image
import numpy as np

# Load heatmap.png as grayscale (ignore color map)
img = Image.open("heatmap.png").convert("L")
arr = np.array(img).astype(np.float32)

# Apply the exact same transformation
disp_gray = (arr / 192.0 * 255.0).clip(0, 255).astype(np.uint8)

# Save result
img_gray = Image.fromarray(disp_gray, mode="L")
img_gray.save("heatmap_transformed.png")


/tmp/ipykernel_324152/2742350039.py:12: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img_gray = Image.fromarray(disp_gray, mode="L")


In [64]:
# Load the image
img = Image.open("heatmap_transformed.png").convert("L")  # force grayscale
arr = np.array(img).astype(np.float32)

# Check min and max
print("Min:", arr.min())
print("Max:", arr.max())

Min: 0.0
Max: 134.0


In [65]:
# Load the image
img = Image.open("depth_gray_gray.png").convert("L")  # force grayscale
arr = np.array(img).astype(np.float32)

# Check min and max
print("Min:", arr.min())
print("Max:", arr.max())

Min: 1.0
Max: 231.0
